# Fase 5 - Pronóstico de Series Temporales: Demanda de Taxis NYC

**Pregunta de negocio:** ¿Podemos predecir la demanda de taxis de la próxima semana?

## Objetivos de este notebook

1. Cargar conteos diarios de viajes para 2015 desde BigQuery
2. Dividir en entrenamiento (enero-octubre) y prueba (noviembre-diciembre)
3. Ajustar modelo ARIMA usando análisis ACF/PACF
4. Ajustar modelo SARIMA con estacionalidad semanal
5. Ajustar modelo Prophet con feriados de EE.UU.
6. Comparar pronósticos de todos los modelos vs valores reales
7. Evaluar con métricas MAPE, MAE y RMSE
8. Generar pronóstico a 14 días con bandas de confianza

## ¿Por qué importa el pronóstico de demanda?

Anticipar cuántos viajes habrá en los próximos días permite a las empresas de taxis:
- **Planificar turnos** de conductores adecuadamente
- **Optimizar precios** (surge pricing) en períodos de alta demanda
- **Reducir tiempos de espera** del pasajero al tener la flota posicionada
- **Gestionar mantenimiento** de vehículos en días de baja demanda

## 1. Configuración e importaciones

In [ ]:
# Conexión a BigQuery
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

In [ ]:
# Librerías de análisis y visualización
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Series temporales
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose

# Prophet
from prophet import Prophet

# Métricas
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Para guardar modelos
import pickle
import os

# Configuración de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 2. Carga de conteos diarios de viajes

Consultamos BigQuery para obtener el número total de viajes por día durante todo 2015.
Esta serie temporal será la base de nuestros modelos de pronóstico.

In [ ]:
query = """
SELECT
    DATE(pickup_datetime) AS fecha,
    COUNT(*) AS trip_count,
    AVG(fare_amount) AS avg_fare,
    AVG(trip_distance) AS avg_distance
FROM
    `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE
    pickup_datetime BETWEEN '2015-01-01' AND '2015-12-31'
    AND fare_amount > 0
    AND fare_amount < 500
    AND trip_distance > 0
GROUP BY
    fecha
ORDER BY
    fecha
"""

# Intentar cargar desde cache, si no existe consultar BigQuery
cache_path = '../../../data/processed/nyc_taxi_daily_2015.csv'
os.makedirs(os.path.dirname(cache_path), exist_ok=True)

if os.path.exists(cache_path):
    df_daily = pd.read_csv(cache_path, parse_dates=['fecha'])
    print(f"Datos cargados desde cache: {len(df_daily)} días")
else:
    df_daily = bq.query_to_df(query)
    df_daily['fecha'] = pd.to_datetime(df_daily['fecha'])
    df_daily.to_csv(cache_path, index=False)
    print(f"Datos descargados de BigQuery y guardados: {len(df_daily)} días")

df_daily = df_daily.set_index('fecha').sort_index()
df_daily.index.freq = 'D'  # Asegurar frecuencia diaria

print(f"\nRango: {df_daily.index.min()} a {df_daily.index.max()}")
print(f"Días con datos: {len(df_daily)}")
df_daily.head(10)

In [ ]:
# Visualización de la serie temporal completa
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Viajes diarios
axes[0].plot(df_daily.index, df_daily['trip_count'], color='#2196F3', linewidth=0.8)
axes[0].set_title('Viajes Diarios de Taxi Amarillo - NYC 2015', fontweight='bold')
axes[0].set_ylabel('Número de viajes')
axes[0].axhline(df_daily['trip_count'].mean(), color='red', linestyle='--', 
                alpha=0.7, label=f"Media: {df_daily['trip_count'].mean():,.0f}")
axes[0].legend()

# Media móvil 7 días
rolling_7 = df_daily['trip_count'].rolling(window=7).mean()
rolling_30 = df_daily['trip_count'].rolling(window=30).mean()
axes[1].plot(df_daily.index, df_daily['trip_count'], color='lightblue', linewidth=0.5, alpha=0.7, label='Diario')
axes[1].plot(df_daily.index, rolling_7, color='#FF5722', linewidth=1.5, label='Media móvil 7 días')
axes[1].plot(df_daily.index, rolling_30, color='#4CAF50', linewidth=2, label='Media móvil 30 días')
axes[1].set_title('Tendencia con Medias Móviles', fontweight='bold')
axes[1].set_ylabel('Número de viajes')
axes[1].set_xlabel('Fecha')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nEstadísticas descriptivas de viajes diarios:")
print(df_daily['trip_count'].describe().apply(lambda x: f'{x:,.0f}'))

## 3. División train/test

Usamos **enero-octubre** (304 días) para entrenamiento y **noviembre-diciembre** (61 días) para prueba.

En series temporales, a diferencia de datos tabulares, **nunca se mezcla aleatoriamente**.
El split siempre es cronológico para respetar la estructura temporal de los datos.

In [ ]:
# Serie principal: conteo de viajes diarios
ts = df_daily['trip_count'].copy()

# Split cronológico
train = ts[:'2015-10-31']
test = ts['2015-11-01':]

print(f"Conjunto de entrenamiento: {train.index.min()} a {train.index.max()} ({len(train)} días)")
print(f"Conjunto de prueba:        {test.index.min()} a {test.index.max()} ({len(test)} días)")

# Visualizar el split
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(train.index, train.values, color='#2196F3', label='Entrenamiento (Ene-Oct)')
ax.plot(test.index, test.values, color='#FF5722', label='Prueba (Nov-Dic)')
ax.axvline(pd.Timestamp('2015-11-01'), color='black', linestyle='--', linewidth=2, label='Punto de corte')
ax.set_title('División Train/Test - Serie Temporal de Demanda', fontweight='bold')
ax.set_ylabel('Viajes diarios')
ax.set_xlabel('Fecha')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Análisis de estacionariedad y componentes

Antes de ajustar modelos ARIMA, necesitamos verificar si la serie es **estacionaria**
(media y varianza constantes en el tiempo). Usamos:
- **Test de Dickey-Fuller Aumentado (ADF):** H0 = la serie tiene raíz unitaria (no estacionaria)
- **Descomposición estacional:** separar tendencia, estacionalidad y residuos

In [ ]:
# Test ADF sobre la serie original
def test_stationarity(series, title='Serie'):
    """Ejecuta el test ADF e imprime resultados."""
    result = adfuller(series.dropna(), autolag='AIC')
    print(f"Test ADF para: {title}")
    print(f"  Estadístico ADF: {result[0]:.4f}")
    print(f"  p-value:         {result[1]:.6f}")
    print(f"  Lags usados:     {result[2]}")
    print(f"  Valores críticos:")
    for key, value in result[4].items():
        print(f"    {key}: {value:.4f}")
    if result[1] < 0.05:
        print("  --> La serie ES estacionaria (rechazamos H0)")
    else:
        print("  --> La serie NO es estacionaria (no rechazamos H0)")
    print()
    return result[1]

# Test sobre serie original
p_original = test_stationarity(train, 'Serie original')

# Test sobre primera diferencia
train_diff = train.diff().dropna()
p_diff = test_stationarity(train_diff, 'Primera diferencia (d=1)')

In [ ]:
# Descomposición estacional (periodo = 7 días para estacionalidad semanal)
decomposition = seasonal_decompose(train, model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(16, 12))

decomposition.observed.plot(ax=axes[0], color='#2196F3')
axes[0].set_title('Serie Original (Observada)', fontweight='bold')
axes[0].set_ylabel('Viajes')

decomposition.trend.plot(ax=axes[1], color='#FF5722')
axes[1].set_title('Tendencia', fontweight='bold')
axes[1].set_ylabel('Viajes')

decomposition.seasonal.plot(ax=axes[2], color='#4CAF50')
axes[2].set_title('Estacionalidad Semanal (periodo=7)', fontweight='bold')
axes[2].set_ylabel('Viajes')

decomposition.resid.plot(ax=axes[3], color='#9C27B0')
axes[3].set_title('Residuos', fontweight='bold')
axes[3].set_ylabel('Viajes')

plt.suptitle('Descomposición Estacional de la Serie de Demanda', 
             fontweight='bold', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

print("La estacionalidad semanal es claramente visible:")
print("los fines de semana tienen menos viajes que los días laborables.")

## 5. Análisis ACF/PACF para determinar órdenes p, d, q

- **ACF (Autocorrelación):** muestra la correlación de la serie consigo misma a diferentes rezagos.
  Ayuda a determinar el orden **q** (media móvil).
- **PACF (Autocorrelación Parcial):** muestra la correlación directa eliminando efectos intermedios.
  Ayuda a determinar el orden **p** (autorregresivo).
- **d** se determina por el número de diferenciaciones necesarias para estacionariedad.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ACF y PACF de la serie original
plot_acf(train.dropna(), lags=40, ax=axes[0, 0], title='ACF - Serie Original')
plot_pacf(train.dropna(), lags=40, ax=axes[0, 1], title='PACF - Serie Original', method='ywm')

# ACF y PACF de la primera diferencia
plot_acf(train_diff.dropna(), lags=40, ax=axes[1, 0], title='ACF - Primera Diferencia')
plot_pacf(train_diff.dropna(), lags=40, ax=axes[1, 1], title='PACF - Primera Diferencia', method='ywm')

# Resaltar lags 7, 14, 21, 28 (estacionalidad semanal)
for ax_row in axes:
    for ax in ax_row:
        for lag in [7, 14, 21, 28]:
            ax.axvline(lag, color='red', alpha=0.2, linestyle=':')

plt.suptitle('Funciones de Autocorrelación para Determinar Órdenes ARIMA',
             fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Observaciones del ACF/PACF:")
print("- Picos significativos en lags 7, 14, 21... confirman estacionalidad semanal")
print("- La primera diferencia muestra estacionariedad -> d=1")
print("- PACF sugiere p=1 o p=2 (decae rápidamente)")
print("- ACF sugiere q=1 (corte abrupto tras lag 1 en serie diferenciada)")

## 6. Modelo ARIMA

**ARIMA(p, d, q)** combina tres componentes:
- **AR(p):** Autorregresivo - usa los p valores pasados
- **I(d):** Integrado - número de diferenciaciones
- **MA(q):** Media Móvil - usa los q errores pasados

Basándonos en el análisis ACF/PACF, probamos ARIMA(2, 1, 1) como punto de partida.

In [ ]:
# Ajustar modelo ARIMA
print("Ajustando modelo ARIMA(2, 1, 1)...")
arima_model = ARIMA(train, order=(2, 1, 1))
arima_result = arima_model.fit()

print(arima_result.summary())

In [ ]:
# Pronóstico ARIMA para noviembre-diciembre
arima_forecast = arima_result.forecast(steps=len(test))
arima_forecast.index = test.index

# Visualizar
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(train[-60:].index, train[-60:].values, color='#2196F3', label='Entrenamiento (últimos 60 días)')
ax.plot(test.index, test.values, color='#4CAF50', linewidth=2, label='Real (Nov-Dic)')
ax.plot(arima_forecast.index, arima_forecast.values, color='#FF5722', 
        linewidth=2, linestyle='--', label='Pronóstico ARIMA(2,1,1)')
ax.set_title('Pronóstico ARIMA vs Valores Reales', fontweight='bold')
ax.set_ylabel('Viajes diarios')
ax.set_xlabel('Fecha')
ax.legend()
plt.tight_layout()
plt.show()

# Métricas
mae_arima = mean_absolute_error(test, arima_forecast)
rmse_arima = np.sqrt(mean_squared_error(test, arima_forecast))
mape_arima = np.mean(np.abs((test - arima_forecast) / test)) * 100

print(f"\nMétricas ARIMA(2,1,1):")
print(f"  MAE:  {mae_arima:,.0f} viajes")
print(f"  RMSE: {rmse_arima:,.0f} viajes")
print(f"  MAPE: {mape_arima:.2f}%")
print(f"\nNota: ARIMA sin componente estacional pierde el patrón semanal.")

## 7. Modelo SARIMA (Seasonal ARIMA)

**SARIMAX(p, d, q)(P, D, Q, s)** extiende ARIMA con componentes estacionales:
- **(P, D, Q):** Órdenes estacionales (autorregresivo, diferenciación, media móvil)
- **s = 7:** Período estacional (semanal)

Esto permite capturar el patrón repetitivo donde los viernes tienen más viajes
que los domingos, semana tras semana.

In [ ]:
# Ajustar modelo SARIMA
print("Ajustando modelo SARIMA(2,1,1)(1,1,1,7)...")
print("Esto puede tardar unos minutos...\n")

sarima_model = SARIMAX(
    train,
    order=(2, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_result = sarima_model.fit(disp=False)

print(sarima_result.summary())

In [ ]:
# Pronóstico SARIMA con intervalos de confianza
sarima_pred = sarima_result.get_forecast(steps=len(test))
sarima_forecast = sarima_pred.predicted_mean
sarima_forecast.index = test.index
sarima_ci = sarima_pred.conf_int()
sarima_ci.index = test.index

# Visualizar con bandas de confianza
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(train[-60:].index, train[-60:].values, color='#2196F3', label='Entrenamiento')
ax.plot(test.index, test.values, color='#4CAF50', linewidth=2, label='Real')
ax.plot(sarima_forecast.index, sarima_forecast.values, color='#FF5722', 
        linewidth=2, linestyle='--', label='Pronóstico SARIMA')
ax.fill_between(sarima_ci.index, 
                sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1],
                color='#FF5722', alpha=0.15, label='IC 95%')
ax.set_title('Pronóstico SARIMA(2,1,1)(1,1,1,7) con Intervalos de Confianza', fontweight='bold')
ax.set_ylabel('Viajes diarios')
ax.set_xlabel('Fecha')
ax.legend()
plt.tight_layout()
plt.show()

# Métricas
mae_sarima = mean_absolute_error(test, sarima_forecast)
rmse_sarima = np.sqrt(mean_squared_error(test, sarima_forecast))
mape_sarima = np.mean(np.abs((test - sarima_forecast) / test)) * 100

print(f"\nMétricas SARIMA(2,1,1)(1,1,1,7):")
print(f"  MAE:  {mae_sarima:,.0f} viajes")
print(f"  RMSE: {rmse_sarima:,.0f} viajes")
print(f"  MAPE: {mape_sarima:.2f}%")

## 8. Modelo Prophet

**Prophet** (Meta/Facebook) es un modelo aditivo diseñado para series temporales de negocio.
Sus ventajas:
- Maneja automáticamente tendencia, estacionalidad y feriados
- Robusto ante datos faltantes y cambios de tendencia
- Fácil de configurar e interpretar
- Genera intervalos de incertidumbre bayesianos

In [ ]:
# Preparar datos para Prophet (requiere columnas 'ds' y 'y')
train_prophet = pd.DataFrame({
    'ds': train.index,
    'y': train.values
})

test_prophet = pd.DataFrame({
    'ds': test.index,
    'y': test.values
})

# Crear y configurar modelo Prophet
prophet_model = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=0.05,
    seasonality_mode='multiplicative'
)

# Agregar feriados de EE.UU.
prophet_model.add_country_holidays(country_name='US')

# Ajustar modelo
print("Ajustando modelo Prophet...")
prophet_model.fit(train_prophet)
print("Modelo Prophet ajustado.")

In [ ]:
# Pronóstico Prophet
future = prophet_model.make_future_dataframe(periods=len(test), freq='D')
prophet_pred = prophet_model.predict(future)

# Extraer pronóstico para el período de prueba
prophet_test = prophet_pred[prophet_pred['ds'].isin(test.index)]
prophet_forecast = pd.Series(prophet_test['yhat'].values, index=test.index)

# Visualizar pronóstico
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(train[-60:].index, train[-60:].values, color='#2196F3', label='Entrenamiento')
ax.plot(test.index, test.values, color='#4CAF50', linewidth=2, label='Real')
ax.plot(prophet_forecast.index, prophet_forecast.values, color='#9C27B0',
        linewidth=2, linestyle='--', label='Pronóstico Prophet')
ax.fill_between(test.index,
                prophet_test['yhat_lower'].values,
                prophet_test['yhat_upper'].values,
                color='#9C27B0', alpha=0.15, label='IC 80%')
ax.set_title('Pronóstico Prophet con Feriados de EE.UU.', fontweight='bold')
ax.set_ylabel('Viajes diarios')
ax.set_xlabel('Fecha')
ax.legend()
plt.tight_layout()
plt.show()

# Métricas
mae_prophet = mean_absolute_error(test, prophet_forecast)
rmse_prophet = np.sqrt(mean_squared_error(test, prophet_forecast))
mape_prophet = np.mean(np.abs((test - prophet_forecast) / test)) * 100

print(f"\nMétricas Prophet:")
print(f"  MAE:  {mae_prophet:,.0f} viajes")
print(f"  RMSE: {rmse_prophet:,.0f} viajes")
print(f"  MAPE: {mape_prophet:.2f}%")

In [ ]:
# Gráficos de componentes de Prophet
fig = prophet_model.plot_components(prophet_pred)
fig.suptitle('Componentes del Modelo Prophet', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Interpretación de componentes:")
print("- Tendencia: muestra la dirección general de la demanda a lo largo del año")
print("- Estacionalidad semanal: viernes y sábado suelen tener más viajes")
print("- Estacionalidad anual: variación por temporada (verano vs invierno)")
print("- Efecto de feriados: días como Navidad o Acción de Gracias reducen la demanda")

## 9. Comparación de todos los modelos

Visualizamos los tres pronósticos juntos contra los valores reales para
evaluar visualmente cuál captura mejor los patrones.

In [ ]:
# Comparación visual
fig, ax = plt.subplots(figsize=(18, 7))

ax.plot(test.index, test.values, color='black', linewidth=2.5, label='Real', zorder=5)
ax.plot(arima_forecast.index, arima_forecast.values, color='#2196F3', 
        linewidth=1.5, linestyle='--', label='ARIMA(2,1,1)', alpha=0.8)
ax.plot(sarima_forecast.index, sarima_forecast.values, color='#FF5722', 
        linewidth=1.5, linestyle='--', label='SARIMA(2,1,1)(1,1,1,7)', alpha=0.8)
ax.plot(prophet_forecast.index, prophet_forecast.values, color='#9C27B0', 
        linewidth=1.5, linestyle='--', label='Prophet', alpha=0.8)

ax.set_title('Comparación de Pronósticos: ARIMA vs SARIMA vs Prophet', fontweight='bold', fontsize=14)
ax.set_ylabel('Viajes diarios')
ax.set_xlabel('Fecha')
ax.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla comparativa de métricas
metrics_df = pd.DataFrame({
    'Modelo': ['ARIMA(2,1,1)', 'SARIMA(2,1,1)(1,1,1,7)', 'Prophet'],
    'MAE': [mae_arima, mae_sarima, mae_prophet],
    'RMSE': [rmse_arima, rmse_sarima, rmse_prophet],
    'MAPE (%)': [mape_arima, mape_sarima, mape_prophet]
})

# Determinar mejor modelo por cada métrica
metrics_df['Mejor MAE'] = metrics_df['MAE'] == metrics_df['MAE'].min()
metrics_df['Mejor RMSE'] = metrics_df['RMSE'] == metrics_df['RMSE'].min()
metrics_df['Mejor MAPE'] = metrics_df['MAPE (%)'] == metrics_df['MAPE (%)'].min()

print("COMPARACIÓN DE MODELOS DE PRONÓSTICO")
print("=" * 65)
print(metrics_df[['Modelo', 'MAE', 'RMSE', 'MAPE (%)']].to_string(index=False))

# Identificar mejor modelo
best_idx = metrics_df['MAPE (%)'].idxmin()
best_model_name = metrics_df.loc[best_idx, 'Modelo']
print(f"\nMejor modelo por MAPE: {best_model_name}")
print(f"  MAPE = {metrics_df.loc[best_idx, 'MAPE (%)']:.2f}%")

## 10. Pronóstico a 14 días con bandas de confianza

Generamos un pronóstico operativo de 2 semanas hacia adelante usando el mejor modelo.
Las bandas de confianza permiten a los gestores de flota prepararse para
escenarios optimistas y pesimistas.

In [ ]:
# Reajustar SARIMA con todos los datos para pronóstico operativo
print("Reajustando SARIMA con todos los datos de 2015...")
sarima_full = SARIMAX(
    ts,
    order=(2, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_full_result = sarima_full.fit(disp=False)

# Pronóstico 14 días
forecast_14 = sarima_full_result.get_forecast(steps=14)
forecast_14_mean = forecast_14.predicted_mean
forecast_14_ci = forecast_14.conf_int()

# Visualización
fig, ax = plt.subplots(figsize=(16, 6))

# Últimos 30 días reales
ax.plot(ts[-30:].index, ts[-30:].values, color='#2196F3', linewidth=2, 
        marker='o', markersize=4, label='Datos reales (últimos 30 días)')

# Pronóstico
ax.plot(forecast_14_mean.index, forecast_14_mean.values, color='#FF5722', 
        linewidth=2.5, marker='s', markersize=5, label='Pronóstico 14 días')

# Bandas de confianza
ax.fill_between(forecast_14_ci.index,
                forecast_14_ci.iloc[:, 0],
                forecast_14_ci.iloc[:, 1],
                color='#FF5722', alpha=0.2, label='IC 95%')

ax.axvline(ts.index[-1], color='black', linestyle='--', linewidth=1.5, alpha=0.5)
ax.set_title('Pronóstico de Demanda: Próximos 14 Días', fontweight='bold', fontsize=14)
ax.set_ylabel('Viajes diarios')
ax.set_xlabel('Fecha')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

# Tabla de pronóstico
forecast_table = pd.DataFrame({
    'Fecha': forecast_14_mean.index.strftime('%Y-%m-%d (%A)'),
    'Pronóstico': forecast_14_mean.values.astype(int),
    'Límite Inferior': forecast_14_ci.iloc[:, 0].values.astype(int),
    'Límite Superior': forecast_14_ci.iloc[:, 1].values.astype(int)
})
print("\nPronóstico detallado - Próximos 14 días:")
print(forecast_table.to_string(index=False))

## 11. Análisis de residuos del mejor modelo

Los residuos de un buen modelo deben ser:
- **Ruido blanco:** sin autocorrelación significativa
- **Media cero:** sin sesgo sistemático
- **Distribución normal:** idealmente normales (para la validez de intervalos de confianza)

In [ ]:
# Residuos del modelo SARIMA
residuals = sarima_result.resid

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Serie temporal de residuos
axes[0, 0].plot(residuals.index, residuals.values, color='#2196F3', linewidth=0.5)
axes[0, 0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0, 0].set_title('Residuos a lo largo del tiempo', fontweight='bold')
axes[0, 0].set_ylabel('Residuo')

# Histograma de residuos
axes[0, 1].hist(residuals, bins=40, density=True, color='#4CAF50', alpha=0.7, edgecolor='white')
x_range = np.linspace(residuals.min(), residuals.max(), 100)
axes[0, 1].plot(x_range, stats.norm.pdf(x_range, residuals.mean(), residuals.std()),
               color='red', linewidth=2, label='Normal teórica')
axes[0, 1].set_title('Distribución de Residuos', fontweight='bold')
axes[0, 1].legend()

# Q-Q plot
stats.probplot(residuals.dropna(), dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Normalidad)', fontweight='bold')

# ACF de residuos
plot_acf(residuals.dropna(), lags=30, ax=axes[1, 1], title='ACF de Residuos')

plt.suptitle('Diagnóstico de Residuos - Modelo SARIMA', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Test de Ljung-Box para autocorrelación en residuos
from statsmodels.stats.diagnostic import acorr_ljungbox
lb_test = acorr_ljungbox(residuals.dropna(), lags=[7, 14, 21], return_df=True)
print("\nTest de Ljung-Box (H0: no hay autocorrelación en residuos):")
print(lb_test)
print("\np-value > 0.05 indica que los residuos se comportan como ruido blanco.")

## 12. Guardar parámetros del mejor modelo

In [ ]:
# Crear directorio para modelos
model_dir = '../../../models/nyc_taxi/'
os.makedirs(model_dir, exist_ok=True)

# Guardar parámetros del modelo SARIMA
model_params = {
    'model_type': 'SARIMA',
    'order': (2, 1, 1),
    'seasonal_order': (1, 1, 1, 7),
    'aic': sarima_result.aic,
    'bic': sarima_result.bic,
    'metrics_test': {
        'mae': mae_sarima,
        'rmse': rmse_sarima,
        'mape': mape_sarima
    },
    'train_period': {'start': str(train.index.min()), 'end': str(train.index.max())},
    'test_period': {'start': str(test.index.min()), 'end': str(test.index.max())},
    'params': dict(sarima_result.params)
}

with open(os.path.join(model_dir, 'sarima_params.pkl'), 'wb') as f:
    pickle.dump(model_params, f)

# Guardar el modelo completo (reajustado con todos los datos)
with open(os.path.join(model_dir, 'sarima_model.pkl'), 'wb') as f:
    pickle.dump(sarima_full_result, f)

# También guardar comparación de modelos
metrics_df.to_csv(os.path.join(model_dir, 'ts_model_comparison.csv'), index=False)

print(f"Modelos guardados en: {os.path.abspath(model_dir)}")
print(f"  - sarima_params.pkl: Parámetros y métricas")
print(f"  - sarima_model.pkl: Modelo SARIMA completo")
print(f"  - ts_model_comparison.csv: Comparación de modelos")

## 13. Discusión y conclusiones

### Resultados principales

| Modelo | Fortaleza | Debilidad |
|--------|-----------|----------|
| ARIMA | Simple, rápido | No captura estacionalidad semanal |
| SARIMA | Captura estacionalidad | Lento, parámetros manuales |
| Prophet | Automático, maneja feriados | Puede sobreajustar patrones |

### ¿Qué modelo elegir?

- **SARIMA** es generalmente el mejor balance entre precisión e interpretabilidad para esta serie
- **Prophet** es preferible cuando hay muchos feriados y cambios de tendencia
- **ARIMA simple** sirve como baseline pero es insuficiente para datos con estacionalidad

### Limitaciones prácticas

1. **Eventos impredecibles:** Tormentas de nieve, huelgas o pandemias no pueden anticiparse con modelos de series temporales puros
2. **Datos agregados:** Pronosticar a nivel diario total pierde la granularidad por zona y hora
3. **Cambios estructurales:** El mercado de taxis evoluciona (competencia de Uber/Lyft), lo que invalida modelos entrenados con datos históricos
4. **Horizonte de pronóstico:** La precisión se degrada rápidamente más allá de 7-14 días

### Próximos pasos

- Notebook 05: Detección de anomalías para identificar días atípicos
- Notebook 06: Patrones de demanda por zona y hora para reposicionar la flota